# ANGEL — laboratório gratuito de vozes (Kokoro / PT-BR)

Este notebook é apenas para **testes locais na sessão gratuita do Google Colab**. Não usa API paga, chave de serviço, assinatura nem compra de créditos. Não escolha Colab Pro nem aceite qualquer solicitação de pagamento.

Objetivo: verificar se a instalação atual do Kokoro consegue sintetizar português brasileiro e comparar as vozes disponíveis. Nenhuma voz fica aprovada automaticamente para o elenco.

## 1. Iniciar o ambiente

No Colab, mantenha o ambiente padrão gratuito. A instalação pode levar alguns minutos e baixar os modelos necessários.

In [ ]:
import os, sys, shutil, subprocess

print('Python do Colab:', sys.version.split()[0])
print('Preparando um Python 3.12 isolado para o Kokoro...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
uv_bin = shutil.which('uv')
if not uv_bin:
    raise RuntimeError('Não foi possível localizar o instalador uv. Pare aqui e envie esta mensagem.')

subprocess.check_call([uv_bin, 'python', 'install', '3.12'])
venv_dir = '/content/angel-kokoro-venv'
subprocess.check_call([uv_bin, 'venv', '--python', '3.12', venv_dir])
venv_python = os.path.join(venv_dir, 'bin', 'python')

print('Instalando Kokoro e dependências dentro do Python 3.12...')
subprocess.check_call([uv_bin, 'pip', 'install', '--python', venv_python, 'kokoro>=0.9.4', 'soundfile'])

print('Instalando espeak-ng...')
subprocess.run(['bash', '-lc', 'apt-get update -qq && apt-get install -y espeak-ng'], check=True)

subprocess.check_call([venv_python, '-c', 'from kokoro import KPipeline; import soundfile; print("Kokoro importado no Python 3.12 com sucesso")'])
print('\nAmbiente ANGEL pronto. Use as próximas células; elas chamarão o Python 3.12 isolado.')

## 2. Inicializar português brasileiro

O código de idioma `p` é o indicado pelo README oficial do Kokoro para português brasileiro. A disponibilidade real das vozes será confirmada durante a execução.

In [ ]:
import os, subprocess
venv_python = '/content/angel-kokoro-venv/bin/python'
if not os.path.exists(venv_python):
    raise RuntimeError('Execute primeiro a célula 1 para preparar o Python 3.12.')

check_code = "from kokoro import KPipeline; p=KPipeline(lang_code='p'); print('Pipeline PT-BR inicializado com sucesso.')"
subprocess.check_call([venv_python, '-c', check_code])

## 3. Comparar as vozes PT-BR encontradas na documentação do projeto

A lista abaixo contém IDs candidatos documentados no ecossistema Kokoro. Como a disponibilidade pode variar entre versões/modelos, cada teste é protegido: se uma voz falhar, o notebook registra o erro e continua. Não considere nenhuma delas escolhida até ouvir e avaliar.

In [ ]:
import os, subprocess
from IPython.display import Audio, display

venv_python = '/content/angel-kokoro-venv/bin/python'
if not os.path.exists(venv_python):
    raise RuntimeError('Execute primeiro a célula 1 e depois a célula 2.')

runner = r'''import os
import numpy as np
import soundfile as sf
from kokoro import KPipeline

pipeline = KPipeline(lang_code='p')
voice_candidates = ['pf_dora', 'pm_alex', 'pm_santa']
sample_text = ('Porque conseguir sozinho não significa que alguém precise fazer tudo sozinho. '
               'Você demorou. Eu sei.')
os.makedirs('/content/angel_voice_tests', exist_ok=True)
successful = []
for voice_id in voice_candidates:
    print(f'\\n--- Testando {voice_id} ---', flush=True)
    try:
        chunks = list(pipeline(sample_text, voice=voice_id, speed=1.0))
        if not chunks:
            print('Sem áudio retornado.', flush=True)
            continue
        audio = np.concatenate([np.asarray(chunk[2]) for chunk in chunks])
        path = f'/content/angel_voice_tests/{voice_id}.wav'
        sf.write(path, audio, 24000)
        successful.append(path)
        print(f'Áudio salvo: {path}', flush=True)
    except Exception as exc:
        print(f'Falha em {voice_id}: {type(exc).__name__}: {exc}', flush=True)
print('ARQUIVOS_GERADOS=' + '|'.join(successful), flush=True)
'''
runner_path = '/content/angel_kokoro_runner.py'
with open(runner_path, 'w', encoding='utf-8') as handle:
    handle.write(runner)

result = subprocess.run([venv_python, runner_path], text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print(result.stderr[-6000:])
if result.returncode != 0:
    raise RuntimeError('O teste de voz falhou. Confira a mensagem acima antes de continuar.')

line = next((x for x in result.stdout.splitlines() if x.startswith('ARQUIVOS_GERADOS=')), '')
generated_files = line.split('=', 1)[1].split('|') if line else []
for audio_path in generated_files:
    if audio_path and os.path.exists(audio_path):
        print('Reproduzindo:', os.path.basename(audio_path))
        display(Audio(filename=audio_path))
print('Vozes que geraram áudio:', [os.path.basename(x) for x in generated_files])

## 4. Baixar os testes

Execute esta célula depois dos testes. Ela cria um ZIP com os WAVs que foram gerados com sucesso.

In [ ]:
import zipfile
from pathlib import Path

wav_files = list(Path('angel_voice_tests').glob('*.wav'))
if not wav_files:
    print('Nenhum WAV foi gerado. Confira as mensagens da etapa anterior.')
else:
    with zipfile.ZipFile('ANGEL_testes_de_voz_Kokoro.zip', 'w') as archive:
        for file_path in wav_files:
            archive.write(file_path, arcname=file_path.name)
    from google.colab import files
    files.download('ANGEL_testes_de_voz_Kokoro.zip')

## Critérios de avaliação para o elenco ANGEL

- Pronúncia natural em português brasileiro, sem sotaque artificial indesejado.
- Cadência, pausas e respiração convincentes.
- Capacidade de transmitir intenção sem soar excessivamente robótica.
- Consistência entre execuções.
- Licença e condições de uso adequadas para a distribuição futura do anime.

**Limites:** o Colab gratuito pode impor cotas, encerrar sessões e variar a disponibilidade de hardware. O notebook não garante qualidade de dublagem profissional. Não envie gravações pessoais nem tente clonar a voz de pessoas reais. Os testes não autorizam uso comercial automático: a licença do modelo, dos pesos e das vozes precisa ser conferida antes de qualquer lançamento.